# Quantum Objects in Quax

This notebook introduces the core data types in Quax and demonstrates how to construct, manipulate, and combine quantum objects using natural Python syntax.

## Type hierarchy

Quax organizes quantum objects into three categories:

| Category | Types | Description |
|----------|-------|-------------|
| **State** | `StateVector`, `DensityMatrix` | Pure and mixed quantum states |
| **Operator** | `Unitary`, `KrausMap`, `Operator`, `Observable` | Gates and general linear maps |
| **SuperOperator** | `SuperOp`, `Choi`, `PauliLiouville` | Quantum channels in various representations |
| **Measurement** | `QuantumInstrument` | Measurements with both classical outcomes and quantum processes |

## Operator syntax

All quantum objects support a consistent set of binary operations:

| Syntax | Operation | Example |
|--------|-----------|---------|
| `@` | Composition / application | `U @ psi` applies gate `U` to state `psi` |
| `\|` | Tensor product | `X \| Z` creates the 2-qubit operator $X \otimes Z$ |
| `*` | Scalar multiplication | `0.5 * U` scales an operator |
| `**` | Powers | `CZ ** 0.5` computes $\sqrt{CZ}$ |
| `-` | Negation | `-U` negates an operator |

When composing objects of different types (e.g. `Choi @ SuperOp`), the result takes the type of the **left** operand.

## Qudit dimensions

Quax tracks the dimension of each qudit explicitly. A 3-qubit state has `dims=(2,2,2)`, while a qubit-qutrit state has `dims=(2,3)`. All data is stored in **tensor format** — a 2-qubit state vector has shape `(2,2)`, not `(4,)` — enabling efficient tensor network operations. Use the `.matrix` property to get the conventional flattened representation.

## JIT compilation

All Quax operations are compatible with `jax.jit`. Dimensions are treated as static arguments, so changing the number or size of qudits triggers recompilation, but changing state data does not.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax
import jax.numpy as jnp

import quax as qx
from quax.gates import CZ, ISWAP, RX, RZ

## States

Quantum states come in two flavours: **state vectors** $|\psi\rangle$ for pure states and **density matrices** $\rho$ for mixed states. Quax provides convenient constructors for common initial states.

In [ ]:
initial_psi = qx.zero_state_vector(3)
initial_rho = qx.zero_state_matrix(3)
mixed_rho = qx.mixed_state_matrix(2)

In [ ]:
mixed_rho

## Operators

Quax includes a standard gate set. Parameterized gates like `RX(θ)` return a `Unitary` for the given angle.

In [ ]:
RX(jnp.pi / 2)

### Composition

The `@` operator composes unitaries: `A @ B` means "apply B first, then A", matching standard matrix multiplication order.

In [ ]:
CZ @ ISWAP

### Tensor products

The `|` operator forms tensor products, building multi-qubit operators from single-qubit gates.

In [ ]:
RX(jnp.pi / 2) | RZ(jnp.pi / 4)

### Scalar multiplication

Multiplying a `Unitary` by 1.0 preserves its type. Multiplying by any other scalar produces a `KrausMap`, since the result is no longer unitary.

In [ ]:
1.0 * CZ

In [ ]:
0.5 * CZ

### Fractional powers

The `**` operator computes matrix powers via eigendecomposition. This is useful for constructing gates like $\sqrt{CZ}$.

In [ ]:
CZ ** (0.5)

## Applying operators to states

The `@` operator also applies operators to states. Quax automatically handles the distinction between pure and mixed state evolution.

### Pure states

Applying a `Unitary` to a `StateVector` computes $|\psi'\rangle = U|\psi\rangle$.

In [ ]:
initial_psi = qx.zero_state_vector(2)
qx.random_state_vector(dims=(2,), key=jax.random.PRNGKey(0))
CZ @ initial_psi

### Mixed states

Applying a `Unitary` to a `DensityMatrix` computes $\rho' = U\rho U^\dagger$ automatically.

In [ ]:
initial_rho = qx.mixed_state_matrix(2)

CZ @ initial_rho

### Batch operations on ensembles

Quax supports batched operations: applying a single gate to an **ensemble** of states. Ensemble dimensions are leading axes — here we create a `(3, 5)` batch of 2-qubit density matrices and apply CZ to all of them simultaneously.

In [ ]:
size = (3, 5)

rhos = qx.random_density_matrix(rank=2, dims=(2, 2), key=jax.random.PRNGKey(0), size=size)
rho_outs = CZ @ rhos
print(rho_outs)

## Superoperators

Superoperators describe general quantum channels, including noisy processes. Quax supports multiple equivalent representations — `SuperOp` (Liouville), `Choi` (Choi–Jamiołkowski), and `PauliLiouville` (Pauli transfer matrix) — and can convert freely between them.

Any unitary can be promoted to a superoperator. We can also construct random CPTP channels:

In [ ]:
choi = qx.random_choi(dims=((2, 2, 2), (2, 2, 2)), rank=4, key=jax.random.key(1))

In [ ]:
S = qx.unitary_to_superop(CZ)
print(S)

### Common noise channels

Quax includes standard noise channels: depolarizing, amplitude damping (thermal relaxation), bit flip, phase flip, and more.

In [ ]:
S = qx.channels.depolarizing(jnp.array(0.1), dims=(2,))
print(S)

### Converting between representations

All superoperator representations encode the same physical channel. Quax provides explicit conversion functions (`superop_to_choi`, `choi_to_pauli_liouville`, etc.) as well as generic converters (`to_choi`, `to_superop`, `to_pauli_liouville`).

In [ ]:
C = qx.superop_to_choi(S)
print(C)

P = qx.superop_to_pauli_liouville(S)
print(P)

### Composing channels

Channels compose with `@`, just like operators. Composing two depolarizing channels produces a more noisy channel. The output type follows the left operand.

In [ ]:
print(S @ S)

Mixed-type composition is supported. The output type always follows the **left** operand:

- `SuperOp @ Choi` → `SuperOp`
- `Choi @ SuperOp` → `Choi`
- `PauliLiouville @ SuperOp @ Choi` → `PauliLiouville`

In [ ]:
S @ C

In [ ]:
C @ S

In [ ]:
P @ S @ C

### Tensor products of channels

The `|` operator extends to superoperators, allowing you to build multi-qubit noise models from single-qubit channels. Mixed-type tensor products are also supported.

In [ ]:
PxP = P | P
print(PxP)

In [ ]:
PxCxS = P | C | S
print(PxCxS)

## Ensembles

Quax ships with standard gate ensembles. The single-qubit Clifford group is an exact unitary 3-design, while the tetrahedral ensemble forms a 2-design with only 12 elements.

In [ ]:
qx.is_two_design(qx.ensembles.CLIFFORD_ENSEMBLE, atol=1e-2)

In [ ]:
qx.is_two_design(qx.ensembles.TETRAHEDRAL_ENSEMBLE, atol=1e-2)

## Qudit states

All of the above works just as well for **qudits** — quantum systems with $d > 2$ levels. A qutrit ($d=3$) has basis states $|0\rangle$, $|1\rangle$, $|2\rangle$; a quart ($d=4$) has four levels, and so on. Specify the dimension via the `dims` tuple.

In tensor format, a qutrit state vector has shape `(3,)`, a two-qutrit state has shape `(3, 3)`, and a qubit-qutrit state has shape `(2, 3)`.

In [ ]:
# Qutrit states
psi_qutrit = qx.zero_state_vector(dims=(3,))
print("Qutrit |0>:", psi_qutrit)
print("  data shape:", psi_qutrit.data.shape)

# Custom superposition state
psi_sup = qx.StateVector.from_matrix(jnp.array([1, 1, 1], dtype=complex) / jnp.sqrt(3), dims=(3,))
print("\n(|0>+|1>+|2>)/sqrt(3):", psi_sup)

# Qutrit density matrices
rho_qutrit = qx.zero_state_matrix(dims=(3,))
rho_mixed = qx.mixed_state_matrix(dims=(3,))
print("\nMaximally mixed qutrit I/3:")
print(rho_mixed.matrix)

### Mixed-dimension registers

Quax supports registers where each subsystem has a **different** dimension. For example, a qubit-qutrit system has `dims=(2, 3)` and lives in a $2 \times 3 = 6$-dimensional Hilbert space. Gates are applied to individual subsystems via `targeted_apply_unitary`.

In [ ]:
# Qubit-qutrit register |0,0>
psi_mixed_reg = qx.zero_state_vector(dims=(2, 3))
print("Qubit-qutrit register:", psi_mixed_reg)
print("  dims:", psi_mixed_reg.dims)
print("  data shape:", psi_mixed_reg.data.shape, "  (2 x 3 tensor)")

# Two-qutrit register
psi_2qt = qx.zero_state_vector(dims=(3, 3))
print("\nTwo-qutrit register:", psi_2qt)

# Targeted gate application: Hadamard on the qubit, qutrit TX on the qutrit
psi_mixed_reg = qx.targeted_apply_unitary(qx.gates.H, psi_mixed_reg, subsystem=(0,))
psi_mixed_reg = qx.targeted_apply_unitary(qx.gates.TX, psi_mixed_reg, subsystem=(1,))
print("\nAfter H on qubit and TX on qutrit:")
print("  ", psi_mixed_reg.pretty_print())

### Automatic promotion

When you apply a qubit operator to a qutrit (or higher-dimensional) subsystem, Quax **automatically promotes** the operator. The qubit gate acts on the $|0\rangle$–$|1\rangle$ subspace and leaves higher levels unchanged (identity on the complement). This models the realistic situation where qubits are encoded in the lowest two levels of a physical system with additional leakage levels.

You can also promote explicitly with `qx.promote(op, target_dims)`.

In [ ]:
# Apply qubit Hadamard directly to a qutrit — it is auto-promoted to 3x3
psi_qutrit = qx.zero_state_vector(dims=(3,))
psi_h = qx.apply_unitary_to_state_vector(qx.gates.H, psi_qutrit)
print("H|0> (auto-promoted to qutrit):", psi_h.pretty_print())

# Compare with explicit promotion
H_promoted = qx.promote(qx.gates.H, (3,))
psi_h_explicit = qx.apply_unitary_to_state_vector(H_promoted, psi_qutrit)
print("promote(H, (3,))|0>:           ", psi_h_explicit.pretty_print())

# The |2> level is untouched by the promoted gate
psi_level2 = qx.StateVector.from_matrix(jnp.array([0, 0, 1], dtype=complex), dims=(3,))
psi_h_level2 = qx.apply_unitary_to_state_vector(qx.gates.H, psi_level2)
print("\nH|2> (auto-promoted):          ", psi_h_level2.pretty_print(), " — |2> is unchanged")

# Targeted auto-promotion: CNOT on a 2-qutrit register
psi_2qt = qx.zero_state_vector(dims=(3, 3))
psi_2qt = qx.targeted_apply_unitary(qx.gates.H, psi_2qt, subsystem=(0,))
psi_2qt = qx.targeted_apply_unitary(qx.gates.CNOT, psi_2qt, subsystem=(0, 1))
print("\nBell state in qutrit encoding:")
print(psi_2qt.pretty_print())

## Measurements

Measurement operations are essential in quantum circuits, and quax has the ability to model them using the quantum instrument formalism.

In [ ]:
# Create a single-qubit measurement instrument
M = qx.gates.MEASURE()

# Prepare a superposition state: |+⟩ = H|0⟩
psi = qx.apply_unitary_to_state_vector(qx.gates.H, qx.zero_state_vector(dims=(2,)))
rho = qx.promote_state_vector_to_density_matrix(psi)
print("Input state ρ:")
print(rho.pretty_print())

# Apply the measurement — returns outcome density matrices and probabilities
rho_outs, probs = qx.apply_instrument_to_density_matrix(M, rho)
print(f"\nMeasurement outcome probabilities: {probs[0]:.2f} for |0⟩, {probs[1]:.2f} for |1⟩")

# Select an outcome using a random key
key = jax.random.key(0)
rho_out, outcome = qx.select_outcome(rho_outs, probs, key)
print(f"\nSampled outcome: {int(outcome)}")
print("Post-measurement state:")
print(rho_out.pretty_print())

## Visualization

Quax provides built-in utilities for visualizing quantum objects.

To visualize an object, use `qx.plot`

Note: Visualization only works for single objects, not for ensembles.

#### Plot state

In [ ]:
psi = qx.random_state_vector(dims=(2, 3, 2, 2), key=jax.random.key(317))
qx.plot(psi)

#### Plot density matrix

In [ ]:
qx.plot(qx.random_density_matrix(dims=(2, 3), rank=2, key=jax.random.key(317)))

#### Plot unitary

In [ ]:
qx.plot(qx.gates.CZ)

#### Plot qutrit unitary

In [ ]:
qx.plot(qx.gates.TRX12(jnp.pi / 2))

#### Plot superoperator

In [ ]:
qx.plot(qx.random_choi(dims=((2, 2), (2, 2)), rank=2, key=jax.random.key(123)))

#### Plot Quantum Instrument

In [ ]:
qi = qx.gates.MEASURE()
qx.plot(qi)

In [ ]:
qi = qx.instrument_from_confusion_and_transition(
    confusion_matrix=jnp.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]),
    transition_matrix=jnp.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]),
    dims=(3,),
)
qx.plot(qi)